In [ ]:
pip install monai # - работа с медицинскими изображениями

In [ ]:
pip install lifelines # - работа с анализом выживаемости

In [ ]:
#pip install captum - для изображений

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, auc, multilabel_confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from tqdm import tqdm
import warnings
import random
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import autocast, GradScaler
from scipy.ndimage import gaussian_filter

from captum.attr import IntegratedGradients, LayerGradCam, DeepLift

warnings.filterwarnings("ignore")

# ====================== КОНФИГУРАЦИЯ ======================
DATA_CSV = ""
BATCH_SIZE = 4
ACCUM_STEPS = 4
NUM_EPOCHS = 80
PRETRAIN_EPOCHS = 12
EMBED_DIM = 512
NUM_HEADS = 8
LEARNING_RATE = 3.2e-5
PRETRAIN_LR = 2.0e-5
MAX_CHANNELS = 4
PATIENCE = 12
N_MODELS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHT_DECAY = 5e-4
TTA_AUGS = 6
SAMPLE_IDX = 5
OUTPUT_DIR = "plots_multi_task"
os.makedirs(OUTPUT_DIR, exist_ok=True)

writer = SummaryWriter(log_dir="runs/multi_task_diagnosis_optimized_v2")

def load_state_dict_safely(model, checkpoint_path, device):
    state = torch.load(checkpoint_path, map_location=device)
    if 'n_averaged' in state:
        state = {k: v for k, v in state.items() if k != 'n_averaged'}
    if any(k.startswith('module.') for k in state.keys()):
        state = {k.replace('module.', '', 1): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    return model

# ====================== LOSS ФУНКЦИИ ======================
class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, num_tasks=4):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks) - 0.5)
    def forward(self, losses):
        log_vars = self.log_vars.to(losses.device)
        precision = torch.exp(-log_vars)
        return (precision * losses + log_vars).sum()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.10):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()

def cox_loss(risk_scores, survival_times, events):
    risk_scores = torch.clamp(risk_scores, -6.0, 6.0)
    order = torch.argsort(survival_times, descending=True)
    risk_scores = risk_scores[order]
    events = events[order]
    log_cumsum_exp = torch.logcumsumexp(risk_scores + 1e-8, dim=0)
    log_likelihood = (risk_scores - log_cumsum_exp) * events
    return -log_likelihood.sum() / (events.sum() + 1e-8)

# ====================== АУГМЕНТАЦИЯ ======================
def strong_augment(img):
    if random.random() > 0.45: return img
    if random.random() < 0.75:
        dims = [d for d in [1,2,3] if random.random() < 0.65]
        if dims: img = torch.flip(img, dims)
    if random.random() < 0.85:
        gamma = random.uniform(0.72, 1.38)
        img = torch.pow(img.clamp(1e-8), gamma)
    if random.random() < 0.7:
        img = img + torch.randn_like(img) * random.uniform(0.015, 0.05)
    if random.random() < 0.65:
        img = img * random.uniform(0.85, 1.15) + random.uniform(-0.12, 0.12)
    return torch.clamp(img, 0.0, 1.0)

def tta_augment(img, aug_idx):
    if aug_idx == 0: return img
    elif aug_idx == 1: return torch.flip(img, [1])
    elif aug_idx == 2: return torch.flip(img, [2])
    elif aug_idx == 3: return torch.flip(img, [3])
    elif aug_idx == 4: return torch.pow(img.clamp(1e-8), 0.85)
    elif aug_idx == 5: return torch.pow(img.clamp(1e-8), 1.25)
    else: return img

# ====================== МОДЕЛЬ ======================
class ModalityEncoder(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, dropout=0.2):
        super().__init__()
        self.patch_size = 16
        self.patch_embed = nn.Conv3d(1, embed_dim, kernel_size=self.patch_size, stride=self.patch_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*3,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x):
        B = x.shape[0]
        patches = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, patches], dim=1)
        tokens = self.transformer(tokens)
        return tokens[:, 0:1], tokens[:, 1:]

class CrossModalFusion(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, dropout=0.18):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.fusion_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*3,
                                       dropout=dropout, activation='gelu', batch_first=True, norm_first=True),
            num_layers=3
        )
    def forward(self, modality_cls_list):
        tokens = torch.cat(modality_cls_list, dim=1)
        residual = tokens
        tokens = self.norm(tokens)
        attn_out, _ = self.cross_attn(tokens, tokens, tokens)
        tokens = residual + attn_out
        return self.fusion_transformer(tokens)

class BidirectionalMultimodalTransformer(nn.Module):
    def __init__(self, clinical_dim=21, embed_dim=512, num_heads=8, num_diagnosis_classes=5, dropout=0.22):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_mod = 4
        self.clinical_dim = clinical_dim

        self.modality_encoders = nn.ModuleList([
            ModalityEncoder(embed_dim, num_heads, num_layers=4, dropout=dropout) for _ in range(4)
        ])
        self.modality_embed = nn.Parameter(torch.randn(4, 1, embed_dim))
        self.cross_fusion = CrossModalFusion(embed_dim, num_heads, dropout=dropout * 0.9)

        self.clinical_embed = nn.Sequential(
            nn.Linear(clinical_dim, embed_dim), nn.GELU(), nn.Dropout(dropout * 1.25),
            nn.Linear(embed_dim, embed_dim), nn.Dropout(dropout)
        )
        self.clinical_to_mri = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.mri_to_clinical = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_clin = nn.LayerNorm(embed_dim)
        self.norm_mri = nn.LayerNorm(embed_dim)

        self.global_fusion = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*3,
                                       dropout=dropout, activation='gelu', batch_first=True, norm_first=True),
            num_layers=2
        )

        head_dropout = dropout * 1.1
        self.tumor_head = nn.Sequential(nn.Linear(embed_dim, embed_dim//2), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(embed_dim//2, 1))
        self.grade_head = nn.Sequential(nn.Linear(embed_dim, embed_dim//2), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(embed_dim//2, 4))
        self.diagnosis_head = nn.Sequential(nn.Linear(embed_dim, embed_dim//2), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(embed_dim//2, num_diagnosis_classes))
        self.survival_head = nn.Sequential(nn.Linear(embed_dim, embed_dim//2), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(embed_dim//2, 1))

        self.recon_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Linear(256, MAX_CHANNELS)
        )

    def forward(self, mri, clinical=None, modality_mask=None, mask_diagnosis_feature=False):
        B, C, D, H, W = mri.shape
        device = mri.device
        mri = mri.float()

        normalized = torch.zeros_like(mri)
        for c in range(C):
            ch = mri[:, c:c+1]
            max_val = ch.amax(dim=(2, 3, 4), keepdim=True)
            normalized[:, c:c+1] = torch.where(max_val > 0, ch / (max_val + 1e-8), ch)
        mri = normalized

        modality_cls = []
        for c in range(self.num_mod):
            if modality_mask is not None and modality_mask[:, c].sum() == 0:
                modality_cls.append(torch.zeros(B, 1, self.embed_dim, device=device))
                continue
            ch = mri[:, c:c+1]
            cls_token, _ = self.modality_encoders[c](ch)
            cls_token = cls_token + self.modality_embed[c]
            modality_cls.append(cls_token)

        fused_tokens = self.cross_fusion(modality_cls)

        if clinical is not None:
            clin = clinical.clone()
            if mask_diagnosis_feature and self.clinical_dim > 5:
                clin[:, 4] = 0.0
            clin_emb = self.clinical_embed(clin).unsqueeze(1)
            clin_out, _ = self.clinical_to_mri(clin_emb, fused_tokens, fused_tokens)
            clin_out = self.norm_clin(clin_out + clin_emb)
            mri_out, _ = self.mri_to_clinical(fused_tokens, clin_out, clin_out)
            fused_tokens = self.norm_mri(fused_tokens + mri_out)
            all_tokens = torch.cat([fused_tokens, clin_out], dim=1)
        else:
            all_tokens = fused_tokens

        global_out = self.global_fusion(all_tokens)
        cls_out = global_out.mean(dim=1)

        recon_loss = torch.tensor(0.0, device=device)
        if clinical is None:
            target = mri.mean(dim=[2, 3, 4])
            recon = self.recon_head(cls_out)
            recon_loss = F.mse_loss(recon, target)

        tumor_logit = self.tumor_head(cls_out).squeeze(1)
        grade_pred = self.grade_head(cls_out)
        diagnosis_pred = self.diagnosis_head(cls_out)
        risk_score = self.survival_head(cls_out).squeeze(1)
        tumor_pred = torch.sigmoid(tumor_logit)

        return tumor_pred, grade_pred, diagnosis_pred, risk_score, recon_loss, cls_out, tumor_logit

# ====================== DATASET ======================
class BrainTumorDataset(Dataset):
    def __init__(self, csv_path, split="train", test_size=0.12, val_size=0.12, random_state=42, is_pretrain=False,
                 scaler=None, le_diag=None, le_idh=None, le_mgmt=None, le_1p19q=None, le_diagnosis=None):
        self.df = pd.read_csv(csv_path)
        self.split = split
        self.is_pretrain = is_pretrain

        self.df = self.df[self.df['label_grade'] != 1].reset_index(drop=True)
        label_map = {0: 0, 2: 1, 3: 2, 4: 3}
        self.df['label_grade_mapped'] = self.df['label_grade'].map(label_map)

        self.df['timepoint_encoded'] = self.df['timepoint'].map({
            'baseline': 0, 'follow-up-1': 1, 'follow-up-2': 2, 'follow-up-3': 3
        }).fillna(0).astype(int)
        self.df['censored'] = self.df['censored'].fillna(0).astype(int)

        if split == "train":
            self.le_diag = LabelEncoder()
            self.le_idh = LabelEncoder()
            self.le_mgmt = LabelEncoder()
            self.le_1p19q = LabelEncoder()
            self.le_diagnosis = LabelEncoder()
            self.df['diagnosis_encoded'] = self.le_diag.fit_transform(self.df['diagnosis'].astype(str))
            self.df['idh_encoded'] = self.le_idh.fit_transform(self.df['idh_status'].astype(str))
            self.df['mgmt_encoded'] = self.le_mgmt.fit_transform(self.df['mgmt_status'].astype(str))
            self.df['1p19q_encoded'] = self.le_1p19q.fit_transform(self.df['1p19q_status'].astype(str))
            self.df['diagnosis_label'] = self.le_diagnosis.fit_transform(self.df['diagnosis'].astype(str))
        else:
            self.le_diag = le_diag
            self.le_idh = le_idh
            self.le_mgmt = le_mgmt
            self.le_1p19q = le_1p19q
            self.le_diagnosis = le_diagnosis
            self.df['diagnosis_encoded'] = self.le_diag.transform(self.df['diagnosis'].astype(str))
            self.df['idh_encoded'] = self.le_idh.transform(self.df['idh_status'].astype(str))
            self.df['mgmt_encoded'] = self.le_mgmt.transform(self.df['mgmt_status'].astype(str))
            self.df['1p19q_encoded'] = self.le_1p19q.transform(self.df['1p19q_status'].astype(str))
            self.df['diagnosis_label'] = self.le_diagnosis.transform(self.df['diagnosis'].astype(str))

        self.df['age_norm'] = self.df['age'].fillna(self.df['age'].median()) / 100.0
        self.df['sex'] = self.df['sex'].fillna(0).astype(int)

        for col in ['idh_status', 'mgmt_status', '1p19q_status']:
            self.df[f'{col}_known'] = (~self.df[col].isin(['Unknown', 'Not Available', 'nan', None])).astype(int)

        self.df['survive_6m_known'] = self.df['survive_6m'].notna().astype(int)
        self.df['survive_12m_known'] = self.df['survive_12m'].notna().astype(int)
        self.df['survive_24m_known'] = self.df['survive_24m'].notna().astype(int)

        self.df['survive_6m'] = self.df['survive_6m'].fillna(0).astype(int)
        self.df['survive_12m'] = self.df['survive_12m'].fillna(0).astype(int)
        self.df['survive_24m'] = self.df['survive_24m'].fillna(0).astype(int)

        self.clinical_cols = [
            'age_norm', 'sex', 'timepoint_encoded', 'censored', 'diagnosis_encoded',
            'idh_encoded', 'mgmt_encoded', '1p19q_encoded',
            'idh_status_known', 'mgmt_status_known', '1p19q_status_known',
            'survive_6m', 'survive_12m', 'survive_24m',
            'survive_6m_known', 'survive_12m_known', 'survive_24m_known'
        ]

        self.clinical_raw = self.df[self.clinical_cols].values.astype(np.float32)

        if split == "train":
            self.scaler = StandardScaler()
            self.clinical = self.scaler.fit_transform(self.clinical_raw)
        else:
            self.scaler = scaler
            self.clinical = self.scaler.transform(self.clinical_raw)

        self.clinical_dim = self.clinical.shape[1]
        self.num_diagnosis_classes = len(self.le_diagnosis.classes_) if hasattr(self, 'le_diagnosis') else 5

        train_idx, temp_idx = train_test_split(range(len(self.df)), test_size=test_size+val_size, random_state=random_state, stratify=self.df['label_grade_mapped'])
        val_idx, test_idx = train_test_split(temp_idx, test_size=test_size/(test_size+val_size), random_state=random_state, stratify=self.df.iloc[temp_idx]['label_grade_mapped'])

        if split == "train": self.indices = train_idx
        elif split == "val": self.indices = val_idx
        else: self.indices = test_idx

        self.df = self.df.iloc[self.indices].reset_index(drop=True)
        self.clinical = self.clinical[self.indices]
        self.clinical_dim = self.clinical.shape[1] + 4
        print(f"Dataset {split}: {len(self.df)} samples | Clinical dim: {self.clinical_dim} | Diagnosis classes: {self.num_diagnosis_classes}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = torch.load(row['preprocessed_path'], weights_only=False)
        if isinstance(img, dict): img = img.get('img', img.get('sequence', img))
        if img.dim() == 5: img = img[0]

        C = img.shape[0]
        if C < MAX_CHANNELS:
            pad = torch.zeros((MAX_CHANNELS - C, *img.shape[1:]), dtype=torch.float32)
            img = torch.cat([img.float(), pad], dim=0)
        elif C > MAX_CHANNELS:
            img = img[:MAX_CHANNELS]

        if self.split == "train":
            img = strong_augment(img)

        try:
            mod_list = ast.literal_eval(row['modality_mask']) if isinstance(row['modality_mask'], str) else row['modality_mask']
        except:
            mod_list = [1, 1, 1, 1]

        modality_mask = torch.tensor(mod_list[:MAX_CHANNELS] + [0.0]*(MAX_CHANNELS - len(mod_list)), dtype=torch.float32)

        tumor_label = torch.tensor(float(row['has_tumor']), dtype=torch.float32)
        grade_label = torch.tensor(int(row['label_grade_mapped']), dtype=torch.long)
        diagnosis_label = torch.tensor(int(row['diagnosis_label']), dtype=torch.long)

        surv_available = row.get('survival_available', 1)
        surv = row['survival_months'] if surv_available == 1 else 15.0
        surv = 15.0 if pd.isna(surv) or surv < 0 else float(surv)
        surv_label = torch.tensor(min(surv / 120.0, 2.5), dtype=torch.float32)

        clinical = torch.tensor(self.clinical[idx], dtype=torch.float32)
        censored = torch.tensor(float(row['censored']), dtype=torch.float32)

        modality_presence = torch.tensor(mod_list[:4], dtype=torch.float32)
        clinical = torch.cat([clinical, modality_presence])

        if self.is_pretrain:
            mask = torch.ones(MAX_CHANNELS)
            for i in random.sample(range(MAX_CHANNELS), random.randint(2, 3)):
                mask[i] = 0.0
                img[i] = 0.0
            return img, modality_mask, mask
        return img, clinical, modality_mask, tumor_label, grade_label, diagnosis_label, surv_label, censored


# ====================== ОСНОВНАЯ ФУНКЦИЯ ======================
def train_multi_task_diagnosis():
    scaler = GradScaler()

    # ==================== ПРЕТРЕНИНГ ====================
    if PRETRAIN_EPOCHS > 0:
        print("=== PRETRAINING (Masked Autoencoder style) ===")
        pretrain_ds = BrainTumorDataset(DATA_CSV, split="train", is_pretrain=True)
        pretrain_loader = DataLoader(pretrain_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

        model = BidirectionalMultimodalTransformer(clinical_dim=0, num_diagnosis_classes=5, dropout=0.2).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)

        for epoch in range(PRETRAIN_EPOCHS):
            model.train()
            total_loss = 0.0
            for img, mod_mask, mask_pre in tqdm(pretrain_loader, desc=f"Pretrain {epoch+1}/{PRETRAIN_EPOCHS}"):
                img, mod_mask = img.to(DEVICE), mod_mask.to(DEVICE)
                optimizer.zero_grad()
                with autocast():
                    _, _, _, _, recon_loss, _, _ = model(img, None, mod_mask)
                if recon_loss.item() > 0:
                    scaler.scale(recon_loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                    scaler.step(optimizer)
                    scaler.update()
                total_loss += recon_loss.item()
            avg_loss = total_loss / len(pretrain_loader)
            print(f"Pretrain Epoch {epoch+1} | Recon Loss: {avg_loss:.5f}")

        torch.save(model.state_dict(), "pretrained_multi_task.pth")
        print("✅ Pretrained weights saved.\n")

    # ==================== ДАННЫЕ ====================
    train_ds = BrainTumorDataset(DATA_CSV, split="train")
    val_ds = BrainTumorDataset(
        DATA_CSV, split="val",
        scaler=train_ds.scaler,
        le_diag=train_ds.le_diag,
        le_idh=train_ds.le_idh,
        le_mgmt=train_ds.le_mgmt,
        le_1p19q=train_ds.le_1p19q,
        le_diagnosis=train_ds.le_diagnosis
    )

    test_ds = BrainTumorDataset(
        DATA_CSV, split="test",
        scaler=train_ds.scaler,
        le_diag=train_ds.le_diag,
        le_idh=train_ds.le_idh,
        le_mgmt=train_ds.le_mgmt,
        le_1p19q=train_ds.le_1p19q,
        le_diagnosis=train_ds.le_diagnosis
    )

    labels = train_ds.df['label_grade_mapped'].values
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    class_weights[1] *= 1.8
    class_weights[2] *= 2.2
    sampler = WeightedRandomSampler(class_weights[labels], len(labels), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    bagging_configs = [
        {"dropout": 0.20, "lr_mult": 1.00, "weight_decay_mult": 1.0, "name": "baseline"},
        {"dropout": 0.24, "lr_mult": 0.92, "weight_decay_mult": 1.15, "name": "high_dropout"},
        {"dropout": 0.18, "lr_mult": 1.08, "weight_decay_mult": 0.85, "name": "low_dropout"},
        {"dropout": 0.26, "lr_mult": 0.95, "weight_decay_mult": 1.25, "name": "conservative"},
        {"dropout": 0.22, "lr_mult": 1.05, "weight_decay_mult": 0.90, "name": "aggressive"},
    ]

    model_paths = []
    for model_idx in range(N_MODELS):
        config = bagging_configs[model_idx]
        print(f"\n--- Модель {model_idx+1}/{N_MODELS} | {config['name']} ---")
        seed = 42 + model_idx * 100
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

        model = BidirectionalMultimodalTransformer(
            clinical_dim=train_ds.clinical_dim,
            num_diagnosis_classes=train_ds.num_diagnosis_classes,
            dropout=config["dropout"]
        ).to(DEVICE)

        if os.path.exists("pretrained_multi_task.pth"):
            state_dict = torch.load("pretrained_multi_task.pth", map_location=DEVICE)
            model_state = model.state_dict()
            for k in list(state_dict.keys()):
                if 'diagnosis_head' in k or 'clinical' in k: continue
                if k in model_state and model_state[k].shape == state_dict[k].shape:
                    model_state[k] = state_dict[k]
            model.load_state_dict(model_state, strict=False)
            print("    Pretrained weights loaded")

        effective_lr = LEARNING_RATE * config["lr_mult"]
        effective_wd = WEIGHT_DECAY * config["weight_decay_mult"]
        optimizer = optim.AdamW(model.parameters(), lr=effective_lr, weight_decay=effective_wd)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=2)
        plateau_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.65, patience=5, min_lr=8e-7)

        uncertainty_loss = UncertaintyWeightedLoss(num_tasks=4).to(DEVICE)
        focal = FocalLoss(gamma=2.0, alpha=torch.tensor([1.0, 2.8, 3.2, 1.3], device=DEVICE), label_smoothing=0.10)
        criterion_tumor = nn.BCEWithLogitsLoss()
        criterion_diagnosis = nn.CrossEntropyLoss(label_smoothing=0.08)

        best_val_loss_model = float('inf')
        patience_counter = 0
        best_model_state = None

        for epoch in range(NUM_EPOCHS):
            model.train()
            for step, batch in enumerate(tqdm(train_loader, desc=f"Model {model_idx+1} Epoch {epoch+1}/{NUM_EPOCHS}")):
                img, clin, mask, y_t, y_g, y_d, y_s, censored = [x.to(DEVICE) for x in batch]
                with autocast():
                    tumor_pred, grade_pred, _, risk_score, _, _, tumor_logit = model(img, clin, mask)
                    _, _, diagnosis_pred, _, _, _, _ = model(img, clin, mask, mask_diagnosis_feature=True)
                    cox = cox_loss(risk_score, y_s * 120, 1 - censored)
                    loss = uncertainty_loss(torch.stack([
                        criterion_tumor(tumor_logit, y_t),
                        focal(grade_pred, y_g),
                        criterion_diagnosis(diagnosis_pred, y_d),
                        cox
                    ]))
                scaler.scale(loss / ACCUM_STEPS).backward()
                if (step + 1) % ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.2)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            scheduler.step()
            model.eval()
            monitor_loss = 0.0
            with torch.no_grad():
                for img, clin, mask, y_t, y_g, y_d, y_s, censored in val_loader:
                    img, clin, mask, y_t, y_g, y_d, y_s, censored = [x.to(DEVICE) for x in [img, clin, mask, y_t, y_g, y_d, y_s, censored]]
                    with autocast():
                        tumor_pred, grade_pred, diagnosis_pred, risk_score, _, _, tumor_logit = model(img, clin, mask, mask_diagnosis_feature=True)
                        l = uncertainty_loss(torch.stack([
                            criterion_tumor(tumor_logit, y_t),
                            focal(grade_pred, y_g),
                            criterion_diagnosis(diagnosis_pred, y_d),
                            cox_loss(risk_score, y_s * 120, 1 - censored)
                        ]))
                    monitor_loss += l.item()
            monitor_loss /= len(val_loader)
            print(f"Epoch {epoch+1:3d} | MonitorLoss: {monitor_loss:.4f}")

            if monitor_loss < best_val_loss_model:
                best_val_loss_model = monitor_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print("Early stopping triggered.")
                    break
            plateau_scheduler.step(monitor_loss)

        if best_model_state is not None:
            path = f"model_{model_idx}.pth"
            torch.save(best_model_state, path)
            model_paths.append(path)
            print(f"Модель {model_idx+1} завершена. Лучший monitor_loss: {best_val_loss_model:.4f}")

    # ==================== TTA + ИНФЕРЕНС ====================
    print("\n=== TTA + MULTI-TASK ИНФЕРЕНС ===")
    all_models = []
    for path in model_paths:
        m = BidirectionalMultimodalTransformer(clinical_dim=train_ds.clinical_dim, num_diagnosis_classes=train_ds.num_diagnosis_classes, dropout=0.22).to(DEVICE)
        m = load_state_dict_safely(m, path, DEVICE)
        m.eval()
        all_models.append(m)

    all_t_true, all_t_pred = [], []
    all_g_true, all_g_pred, all_g_pred_proba = [], [], []
    all_d_true, all_d_pred, all_d_pred_proba = [], [], []
    all_risk_scores, all_surv_true, all_censored = [], [], []

    with torch.no_grad():
        for img, clin, mask, y_t, y_g, y_d, y_s, censored in tqdm(test_loader, desc="TTA Testing"):
            img, clin, mask = img.to(DEVICE), clin.to(DEVICE), mask.to(DEVICE)
            y_t, y_g, y_d, y_s, censored = y_t.to(DEVICE), y_g.to(DEVICE), y_d.to(DEVICE), y_s.to(DEVICE), censored.to(DEVICE)

            tumor_preds, grade_preds, diag_preds, risk_preds = [], [], [], []
            for aug_idx in range(TTA_AUGS):
                aug_img = tta_augment(img.clone(), aug_idx)
                for m in all_models:
                    tp, gp, dp, rs, _, _, _ = m(aug_img, clin, mask, mask_diagnosis_feature=True)
                    tumor_preds.append(tp); grade_preds.append(gp); diag_preds.append(dp); risk_preds.append(rs)

            tumor_pred = torch.stack(tumor_preds).mean(0)
            grade_pred = torch.stack(grade_preds).mean(0)
            diagnosis_pred = torch.stack(diag_preds).mean(0)
            risk_score = torch.stack(risk_preds).mean(0)

            all_t_pred.extend(tumor_pred.cpu().numpy())
            all_t_true.extend(y_t.cpu().numpy())
            all_g_pred.extend(grade_pred.argmax(1).cpu().numpy())
            all_g_true.extend(y_g.cpu().numpy())
            all_g_pred_proba.extend(torch.softmax(grade_pred, dim=1).cpu().numpy())
            all_d_pred.extend(diagnosis_pred.argmax(1).cpu().numpy())
            all_d_true.extend(y_d.cpu().numpy())
            all_d_pred_proba.extend(torch.softmax(diagnosis_pred, dim=1).cpu().numpy())
            all_risk_scores.extend(risk_score.cpu().numpy())
            all_surv_true.extend(y_s.cpu().numpy())
            all_censored.extend(censored.cpu().numpy())

    # Сохранение ансамбля
    ensemble_checkpoint = {
        "ensemble_state_dicts": [torch.load(p, map_location='cpu') for p in model_paths],
        "clinical_dim": train_ds.clinical_dim,
        "num_diagnosis_classes": train_ds.num_diagnosis_classes,
        "scaler": train_ds.scaler,
        "le_diagnosis": train_ds.le_diagnosis,
    }
    torch.save(ensemble_checkpoint, "brain_tumor_ensemble.pt")
    print(f"✅ Ансамбль сохранён: brain_tumor_ensemble_final.pt")

    # ==================== FINAL RESULTS ====================
    print("\n" + "="*70)
    print("FINAL RESULTS")
    print("="*70)

    acc_t = accuracy_score(all_t_true, np.round(all_t_pred))
    f1_t = f1_score(all_t_true, np.round(all_t_pred), zero_division=0)
    auc_t = roc_auc_score(all_t_true, all_t_pred) if len(set(all_t_true)) > 1 else 0.0
    acc_g = accuracy_score(all_g_true, all_g_pred)
    f1_g = f1_score(all_g_true, all_g_pred, average='macro', zero_division=0)
    acc_d = accuracy_score(all_d_true, all_d_pred)
    f1_d = f1_score(all_d_true, all_d_pred, average='macro', zero_division=0)

    print(f"Tumor     → Acc: {acc_t:.4f} | F1: {f1_t:.4f} | AUC: {auc_t:.4f}")
    print(f"Grade     → Acc: {acc_g:.4f} | Macro F1: {f1_g:.4f}")
    print(f"Diagnosis → Acc: {acc_d:.4f} | Macro F1: {f1_d:.4f}")

    print("\n=== CLASSIFICATION REPORT (Diagnosis) ===")
    print(classification_report(all_d_true, all_d_pred, target_names=train_ds.le_diagnosis.classes_, zero_division=0))

    uncensored = np.array(all_censored) == 0
    if np.sum(uncensored) > 10:
        try:
            from lifelines.utils import concordance_index
            c_index = concordance_index(np.array(all_surv_true)[uncensored], -np.array(all_risk_scores)[uncensored], event_observed=[1]*np.sum(uncensored))
            print(f"\nC-index: {c_index:.4f}")
        except:
            pass

    print("\n=== CLASSIFICATION REPORT (Grade) ===")
    print(classification_report(all_g_true, all_g_pred, labels=[0,1,2,3], target_names=['Grade 0','Grade 2','Grade 3','Grade 4'], zero_division=0))

    # ==================== ПРЕДСКАЗАНИЕ ВЫЖИВАЕМОСТИ ====================
    print("\n=== ПРЕДСКАЗАНИЕ ВЫЖИВАЕМОСТИ ПАЦИЕНТОВ ===")
    risk_scores = np.array(all_risk_scores)
    predicted_survival = 120 / (1 + np.exp(risk_scores * 1.8))

    low_risk = risk_scores < np.percentile(risk_scores, 33)
    medium_risk = (risk_scores >= np.percentile(risk_scores, 33)) & (risk_scores < np.percentile(risk_scores, 66))
    high_risk = risk_scores >= np.percentile(risk_scores, 66)

    print(f"Low Risk:    {np.sum(low_risk):4d} пациентов | Средняя выживаемость: {np.mean(predicted_survival[low_risk]):.1f} мес")
    print(f"Medium Risk: {np.sum(medium_risk):4d} пациентов | Средняя выживаемость: {np.mean(predicted_survival[medium_risk]):.1f} мес")
    print(f"High Risk:   {np.sum(high_risk):4d} пациентов | Средняя выживаемость: {np.mean(predicted_survival[high_risk]):.1f} мес")


if __name__ == "__main__":
    train_multi_task_diagnosis()
    writer.close()